# AquaTwin ML starter kit
Simulated testbed data → digital twin → anomaly detection and hybrid prediction → metrics.

Run all cells (Runtime → Run all). The first cell asks you to pick `aquatwin_ml.zip` from your computer if it isn't already in Colab. The whole notebook takes about 6 minutes.

In [ ]:
# Setup: finds the AquaTwin files wherever they are, or asks you to upload the zip.
import glob, os, zipfile, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wntr", "xgboost"], check=True)

def find_project():
    hits = glob.glob("/content/**/simulate.py", recursive=True)
    return os.path.dirname(hits[0]) if hits else None

root = find_project()
if root is None:
    zips = glob.glob("/content/**/*.zip", recursive=True)
    if not zips:
        from google.colab import files
        print("Select aquatwin_ml.zip from your computer:")
        up = files.upload()
        zips = [os.path.join("/content", n) for n in up if n.endswith(".zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/content")
    root = find_project()

assert root, "Could not find simulate.py. Upload aquatwin_ml.zip and run this cell again."
os.chdir(root)
sys.path.insert(0, root)
print("Working in", root)
print(sorted(os.listdir(root)))

In [ ]:
# Regenerate the 30 h dataset (skip this cell to use the included data/sensor_data.csv)
!python simulate.py

In [ ]:
!python pipeline.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/report.md').read()))

In [ ]:
import glob
for f in sorted(glob.glob('results/run*_twin_vs_measured.png')):
    display(Image(f))

## Try it on one sample
Load the saved hybrid model and predict what outlet turbidity *should* be right now.

In [ ]:
import joblib, pandas as pd
import pipeline as P
df = P.add_twin(P.clean(pd.read_csv('data/sensor_data.csv')), 'data/twin_predictions.csv')
bundle = joblib.load('results/predict_turb_out_NTU_random_forest.joblib')
X = P.prediction_frame(df)[bundle['features']]
i = 80000
expected = df.twin_turb_out_NTU[i] + bundle['model'].predict(X.iloc[[i]])[0]
print(f"measured {df.turb_out_NTU[i]:.2f} NTU, twin expects {expected:.2f} NTU")